In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')
import datetime
from collections import defaultdict

In [2]:
# train_news_path = '/home/luxiaoling/wukun/MIND/data/large/train/preprocessed_news.tsv'
# val_news_path = '/home/luxiaoling/wukun/MIND/data/large/dev/preprocessed_news.tsv'
train_behavior_path = '/home/luxiaoling/wukun/MIND/data/large/train/processed_behavior.csv'
val_behavior_path = '/home/luxiaoling/wukun/MIND/data/large/dev/processed_behavior.csv'

In [9]:
# train_news_df = pd.read_csv(train_news_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
# val_news_df = pd.read_csv(val_news_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
# train_news_df.head(2)

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...


In [10]:
# val_news_df.head(2)

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...


In [3]:
train_behavior_df = pd.read_csv(train_behavior_path, sep='\t', names=['uid', 'impr_time', 'history', 'cur_impr'])
val_behavior_df = pd.read_csv(val_behavior_path, sep='\t', names=['uid', 'impr_time', 'history', 'cur_impr'])
train_behavior_df.head()

,uid,impr_time,history,cur_impr
0,U403465,2019-11-09 00:00:00,N59850 N104930 N68866 N82374 N123325 N127916 N...,N92613-0 N17456-0 N67369-0 N31486-0 N76810-0 N...
1,U493092,2019-11-09 00:00:02,N3960 N115631 N21659 N79594 N128643 N52938 N12...,N6865-0 N128031-0 N79990-0 N121426-0 N2198-0 N...
2,U172654,2019-11-09 00:00:03,N43253 N119785 N81937 N128965 N58990 N74795 N7...,N65783-1 N119284-0 N76810-0
3,U248125,2019-11-09 00:00:07,N102417 N22902 N95420 N81970 N24696 N115565 N9...,N90042-0 N2598-0 N108809-0 N104610-0 N18956-0 ...
4,U495159,2019-11-09 00:00:13,N88765 N74317 N63723 N121794 N95711 N64341 N75...,N104437-0 N76810-0 N98657-0 N25492-0 N108809-0...


In [4]:
train_behavior_df.shape, val_behavior_df.shape

((2232748, 4), (376471, 4))

In [5]:
val_behavior_df.head()

,uid,impr_time,history,cur_impr
0,U134050,2019-11-15 08:55:22,N12246 N128820 N119226 N4065 N67770 N33446 N10...,N91737-0 N30206-0 N54368-0 N117802-0 N18190-0 ...
1,U254959,2019-11-15 11:42:35,N34011 N9375 N67397 N7936 N118985 N109453 N103...,N119999-0 N24958-0 N104054-0 N33901-0 N9250-0 ...
2,U499841,2019-11-15 09:08:21,N63858 N26834 N6379 N85484 N15229 N65119 N1047...,N18190-0 N89764-0 N91737-0 N54368-0 N49978-1 N...
3,U107107,2019-11-15 05:50:31,N12959 N8085 N18389 N3758 N9740 N90543 N129790...,N122944-1 N18190-0 N55801-0 N59297-0 N128045-0...
4,U492344,2019-11-15 05:02:25,N109183 N48453 N85005 N45706 N98923 N46069 N35...,N64785-0 N82503-0 N32993-0 N122944-0 N29160-0 ...


In [6]:
def get_news_set(df, history=False):
    history_set = set()
    impr_set = set()
    for imprs in df['cur_impr']:
        news = [x[:-2] for x in imprs.split()]
        impr_set.update(news)
    if history:
        for news in df['history']:
            try:
                history_set.update(news.split())
            except Exception as e:
                continue
    return history_set | impr_set
news_set_train = get_news_set(train_behavior_df, history=True)
news_set_val = get_news_set(val_behavior_df, history=True)
all_news = list(news_set_train | news_set_val) + ['NewsPAD']
news2idx = dict(zip(all_news, range(len(all_news))))

In [7]:
len(all_news)

104152

In [8]:
news2idx['N90543']

19666

In [9]:
import json
with open('./data/large/news2idx.json', 'w') as f:
    json.dump(news2idx, f, indent=4)

In [10]:
uid_list = list(train_behavior_df.uid.unique()) + ['UID_UNK', 'UID_PAD']
uid2idx = dict(zip(uid_list, range(len(uid_list))))

In [11]:
uid2idx['U254959']

260095

In [12]:
with open('./data/large/uid2idx.json', 'w') as f:
    json.dump(uid2idx, f, indent=4)

In [13]:
news_set_len = len(all_news)
news_set_len

104152

In [14]:
len(uid2idx)

711224

In [15]:
uid_history = defaultdict(set)
for uid, history in zip(train_behavior_df.uid, train_behavior_df.history):
    try:
        news = history.split()
        news_id = [news2idx[x] for x in news]
        uid_idx = uid2idx[uid] + news_set_len  # offset 便于后面使用deepwalk预训练
        uid_history[uid_idx].update(news_id)
    except Exception as e:
        continue

In [16]:
from tqdm import tqdm

In [17]:
with open('./data/large/train/uid_news.adjlist', 'w') as f:
    for uid, news in tqdm(uid_history.items()):
        for one_news in news:
            line = str(uid) + ' ' +  str(one_news) + '\n'
            f.write(line)

100%|██████████| 698365/698365 [00:37<00:00, 18698.39it/s]


In [18]:
# with open('./data/large/news2idx.json', 'r') as f:
#     news2idx = json.load(f)

# with open('./data/large/uid2idx.json', 'r') as f:
#     uid2idx = json.load(f)

In [20]:
uid_embeddings = np.zeros(shape=(len(uid2idx), 128))
news_embeddings = np.zeros(shape=(len(news2idx), 128))

In [21]:
with open('./data/large/uid_news.embedding', 'r') as f:
    f.readline()
    for line in f:
        fields = line.split()
        vetex = int(fields[0])
        vec = np.array([float(x) for x in fields[1:]])
        if vetex >= news_set_len:
            uid_embeddings[vetex - news_set_len] = vec
        else:
            news_embeddings[vetex] = vec

In [22]:
uid_embeddings[260095]

array([-5.34043130e-02, -4.72576500e-01, -1.90094550e-01,  2.08502190e-01,
        1.69242100e-01, -2.14441220e-01,  3.04251850e-01, -5.58751970e-02,
        1.00044325e-01,  1.45154280e-01,  1.13700110e-01,  3.07849050e-01,
       -1.80465670e-01, -2.61708860e-01, -7.30258450e-02,  4.93868250e-02,
        8.41097400e-02, -2.50429720e-01, -1.88357890e-02, -1.77471800e-03,
       -9.35415600e-03, -3.61347350e-01,  1.15909990e-01,  1.04794530e-02,
        4.52214740e-02, -2.02770400e-01,  6.04016070e-02,  1.05298250e-01,
        2.24838940e-01,  2.49240830e-01,  2.33943760e-02, -2.34544810e-01,
        2.82843580e-02,  1.32335780e-01,  8.57666700e-02,  5.38962100e-02,
       -2.02522670e-01, -1.84731300e-01, -2.27969570e-01,  5.42856270e-02,
        2.14075900e-01, -2.46185850e-01, -6.10910170e-02,  8.12099800e-02,
       -8.93958960e-02, -4.46430000e-02,  2.13083800e-03,  1.77795660e-01,
       -3.41005960e-02,  2.41788010e-01, -1.50759680e-01,  3.04725860e-01,
       -1.24522780e-01, -

In [23]:
news_embeddings[19666]

array([ 2.61000220e-03, -3.10670270e-02, -3.00017330e-01,  4.46495100e-02,
        2.65930130e-02, -1.42944900e-01, -1.41645120e-01, -2.37820220e-01,
        1.50628240e-01,  3.08016660e-01, -9.34840440e-02, -2.09350910e-01,
        3.74366430e-01,  3.41559150e-03,  1.66596170e-01,  1.16145380e-01,
        1.99582650e-02, -1.43298040e-01, -5.76497430e-02, -2.66113580e-01,
        9.11768700e-02, -2.21860630e-01, -3.80222100e-02,  3.50314400e-01,
       -2.24455570e-01, -5.73495900e-02, -3.58664160e-01, -7.18035800e-02,
       -5.51410160e-03,  3.47978500e-02, -1.35765520e-01,  1.46042400e-01,
       -3.36840150e-02, -3.61818860e-02, -2.04931520e-02,  6.39491300e-02,
       -2.33978460e-02, -3.01325140e-01,  4.23822630e-02, -1.38450770e-01,
        9.77144900e-02,  1.22885520e-01,  1.32166850e-01,  1.87306910e-01,
       -1.54121900e-01,  1.25092740e-01,  2.25311710e-01,  4.10524300e-02,
        5.08991780e-02, -1.23873040e-03, -3.43384900e-01,  2.07977180e-01,
        1.29871170e-01, -

In [ ]:
np.save('./data/large/uid_embeddings_deepwalk.npy', )